# Wearable Fatigue Detection — Full Pipeline from Raw Files

Works directly from your raw session files — `all_sensors_data_1.csv`, `all_sensors_data_2.csv`, `ecg_raw_data_1.csv`, `ecg_raw_data_2.csv`. No separate preprocessing step needed.

The ECG processing here ports the validated Pan-Tompkins-style pipeline (proper gap handling around WiFi stalls, adaptive-threshold R-peak detection, windowed HR/SDNN/RMSSD with outlier rejection) instead of a simple fixed-threshold detector — this is what actually produced a real 0.72 correlation with your fatigue labels in session 1, versus ~0 for the raw on-device ecg_bpm column.

**Upload all 4 files together in Cell 2.**

## 1. Setup & upload

In [ ]:
!pip install -q scikit-learn scipy pandas numpy matplotlib joblib

In [ ]:
import sys, io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from scipy.signal import butter, filtfilt

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import files
    print("Upload: all_sensors_data_1.csv, all_sensors_data_2.csv, ecg_raw_data_1.csv, ecg_raw_data_2.csv (all at once)")
    uploaded = files.upload()
else:
    import glob
    uploaded = {}
    for fpath in glob.glob('all_sensors_data_*.csv') + glob.glob('ecg_raw_data_*.csv'):
        with open(fpath, 'rb') as fh:
            uploaded[fpath] = fh.read()

print(f"\nLoaded {len(uploaded)} file(s): {list(uploaded.keys())}")

## 2. Pair sessions by id (e.g. 'data_1', 'data_2')

In [ ]:
def extract_session_id(filename):
    name = filename
    if name.endswith('.csv'):
        name = name[:-4]
    for prefix in ('all_sensors_', 'ecg_raw_'):
        if name.startswith(prefix):
            name = name[len(prefix):]
            break
    return name if name else None

summary_files, raw_files = {}, {}
for fname in uploaded:
    sid = extract_session_id(fname)
    if sid is None:
        continue
    if fname.startswith('all_sensors'):
        summary_files[sid] = fname
    elif fname.startswith('ecg_raw'):
        raw_files[sid] = fname

session_ids = sorted(set(summary_files) & set(raw_files))
print(f"Complete sessions: {len(session_ids)} → {session_ids}")

## 3. ECG processing functions
Ported from the validated `ecg_reprocess.py` pipeline: uniform resampling with gap-flagging, Pan-Tompkins-style adaptive R-peak detection run separately per gap-free segment, and windowed HR/SDNN/RMSSD with RR outlier rejection.

In [ ]:
def resample_uniform(df, target_fs=50, max_gap_ms=150):
    """Uniform target_fs Hz grid. Real stalls (>max_gap_ms) are flagged invalid, never interpolated across."""
    t0, t1 = df['timestamp'].iloc[0], df['timestamp'].iloc[-1]
    n = int((t1 - t0).total_seconds() * target_fs) + 1
    grid = pd.date_range(t0, periods=n, freq=f'{1000/target_fs:.6f}ms')

    orig_t = df['timestamp'].values.astype('int64') / 1e9
    grid_t = grid.values.astype('int64') / 1e9
    ecg_interp = np.interp(grid_t, orig_t, df['raw_ecg'].values)

    gap_mask = np.zeros(len(grid_t), dtype=bool)
    orig_gaps_ms = np.diff(orig_t) * 1000
    for i in np.where(orig_gaps_ms > max_gap_ms)[0]:
        lo, hi = orig_t[i], orig_t[i+1]
        gap_mask |= (grid_t > lo) & (grid_t < hi)

    marked_idx = np.clip(np.searchsorted(orig_t, grid_t, side='right') - 1, 0, len(df)-1)
    marked_grid = df['marked'].values[marked_idx]

    return pd.DataFrame({'timestamp': grid, 'raw_ecg': ecg_interp, 'valid': ~gap_mask, 'marked': marked_grid})


def bandpass(sig, fs, low=0.5, high=20.0, order=3):
    nyq = fs / 2
    b, a = butter(order, [low/nyq, min(high/nyq, 0.99)], btype='band')
    return filtfilt(b, a, sig)


def detect_r_peaks(grid, fs):
    """Simplified Pan-Tompkins detector, run separately on each contiguous valid chunk."""
    peak_times = []
    grid = grid.copy()
    grid['chunk'] = (~grid['valid']).cumsum()

    for _, seg in grid[grid['valid']].groupby('chunk'):
        if len(seg) < fs * 2:
            continue
        sig = seg['raw_ecg'].to_numpy(dtype=float)
        t = seg['timestamp'].to_numpy()

        filtered = bandpass(sig, fs)
        deriv = np.gradient(filtered) * fs
        squared = deriv ** 2
        win = max(1, int(0.150 * fs))
        integrated = np.convolve(squared, np.ones(win)/win, mode='same')

        refractory_s = 0.25
        spk = np.max(integrated[:fs]) * 0.5
        npk = np.mean(integrated[:fs]) * 0.5
        last_peak_t = None

        for i in range(1, len(integrated)-1):
            threshold = npk + 0.25 * (spk - npk)
            is_local_max = integrated[i] > integrated[i-1] and integrated[i] >= integrated[i+1]
            if integrated[i] > threshold and is_local_max:
                cur_t = t[i]
                if last_peak_t is None or (cur_t - last_peak_t) / np.timedelta64(1, 's') > refractory_s:
                    peak_times.append(cur_t)
                    last_peak_t = cur_t
                    spk = 0.125 * integrated[i] + 0.875 * spk
                    continue
            npk = 0.125 * integrated[i] + 0.875 * npk

    return pd.Series(pd.to_datetime(peak_times)).sort_values().reset_index(drop=True)


def windowed_hrv(peak_times, grid, window_s=30, step_s=5):
    rows = []
    t0, t1 = grid['timestamp'].iloc[0], grid['timestamp'].iloc[-1]
    cur = t0 + pd.Timedelta(seconds=window_s)

    while cur <= t1:
        win_start = cur - pd.Timedelta(seconds=window_s)
        pk = peak_times[(peak_times > win_start) & (peak_times <= cur)]
        rr = np.diff(pk.values).astype('timedelta64[ms]').astype(float)
        rr = rr[(rr > 300) & (rr < 2000)]

        if len(rr) >= 3:
            hr = 60000 / np.median(rr)
            sdnn = np.std(rr)
            rmssd = np.sqrt(np.mean(np.diff(rr)**2)) if len(rr) >= 4 else np.nan
        else:
            hr = sdnn = rmssd = np.nan

        m = grid[(grid['timestamp'] > win_start) & (grid['timestamp'] <= cur)]['marked']
        marked_majority = int(m.mean() >= 0.5) if len(m) else np.nan

        rows.append({'timestamp': cur, 'n_beats': len(rr), 'hr_bpm': hr,
                     'hrv_sdnn_ms': sdnn, 'hrv_rmssd_ms': rmssd, 'marked': marked_majority})
        cur += pd.Timedelta(seconds=step_s)

    return pd.DataFrame(rows)

## 4. Run ECG processing per session, then merge with motion/temp

In [ ]:
TARGET_FS = 50
WINDOW_S = 30
STEP_S = 5

session_frames = []

for sid in session_ids:
    raw = pd.read_csv(io.BytesIO(uploaded[raw_files[sid]]))
    raw['timestamp'] = pd.to_datetime(raw['timestamp'], format='mixed')
    raw = raw.sort_values('timestamp').reset_index(drop=True)

    grid = resample_uniform(raw, target_fs=TARGET_FS)
    peaks = detect_r_peaks(grid, TARGET_FS)
    ecg_feats = windowed_hrv(peaks, grid, window_s=WINDOW_S, step_s=STEP_S)

    summary = pd.read_csv(io.BytesIO(uploaded[summary_files[sid]]))
    summary['timestamp'] = pd.to_datetime(summary['timestamp'], format='mixed')
    summary = summary.sort_values('timestamp')

    merged = pd.merge_asof(
        ecg_feats.sort_values('timestamp'),
        summary[['timestamp', 'motion_magnitude', 'skin_temp', 'ppg_bpm']],
        on='timestamp', direction='nearest', tolerance=pd.Timedelta('5s')
    )
    merged['session_id'] = sid

    valid = merged.dropna(subset=['hr_bpm', 'marked'])
    if len(valid) >= 5:
        print(f"Session {sid}: {len(peaks)} R-peaks detected, {len(merged)} windows "
              f"({len(valid)} usable). hr_bpm corr with marked: {valid['hr_bpm'].corr(valid['marked']):+.3f}")
    else:
        print(f"Session {sid}: {len(peaks)} R-peaks detected, {len(merged)} windows (too few usable to report correlation)")

    session_frames.append(merged)

df = pd.concat(session_frames, ignore_index=True)
print(f"\nCombined total: {len(df)} rows across {len(session_ids)} session(s)")
df.head()

## 5. Clean & prepare features

In [ ]:
FEATURE_COLS = ['hr_bpm', 'hrv_sdnn_ms', 'hrv_rmssd_ms', 'motion_magnitude', 'skin_temp', 'ppg_bpm']

before = len(df)
df = df.dropna(subset=FEATURE_COLS + ['marked'])
after = len(df)
print(f"Kept {after} of {before} rows ({before-after} dropped — mostly windows with too few detected beats)")
print()
print('Rows and marked-rate per session:')
print(df.groupby('session_id')['marked'].agg(['count', 'mean']))

## 6. Split by session

In [ ]:
TARGET_COL = 'marked'
print(f"Available sessions: {session_ids}")
TEST_SESSION_IDS = [session_ids[-1]]  # change to pick a different held-out session
print(f"Testing on: {TEST_SESSION_IDS}  |  Training on: {[s for s in session_ids if s not in TEST_SESSION_IDS]}")

train_mask = ~df['session_id'].isin(TEST_SESSION_IDS)
test_mask = df['session_id'].isin(TEST_SESSION_IDS)

X_train, X_test = df.loc[train_mask, FEATURE_COLS], df.loc[test_mask, FEATURE_COLS]
y_train = df.loc[train_mask, TARGET_COL].astype(int)
y_test = df.loc[test_mask, TARGET_COL].astype(int)

print(f'\nTrain: {X_train.shape}, positive rate {y_train.mean():.2%}')
print(f'Test:  {X_test.shape}, positive rate {y_test.mean():.2%}')

## 7. Train & evaluate

In [ ]:
model = RandomForestClassifier(n_estimators=200, max_depth=5, class_weight='balanced', random_state=42)
model.fit(X_train, y_train)

train_acc = accuracy_score(y_train, model.predict(X_train))
test_acc = accuracy_score(y_test, model.predict(X_test))
majority_baseline = max(y_test.mean(), 1 - y_test.mean())

print(f'Training accuracy:      {train_acc:.2f}')
print(f'Test accuracy:          {test_acc:.2f}')
print(f'Trivial baseline would: {majority_baseline:.2f}')
print('\u2713 Beats trivial baseline' if test_acc > majority_baseline + 0.03 else '\u26a0 Barely above trivial baseline')

if y_test.nunique() > 1:
    print(f'ROC-AUC: {roc_auc_score(y_test, model.predict_proba(X_test)[:,1]):.3f}')

print('\nClassification report:')
print(classification_report(y_test, model.predict(X_test), target_names=['normal','fatigued'], zero_division=0))
print('Confusion matrix:')
print(confusion_matrix(y_test, model.predict(X_test)))

## 8. Feature importance

In [ ]:
importances = model.feature_importances_
order = np.argsort(importances)[::-1]

plt.figure(figsize=(7,4))
plt.bar(range(len(FEATURE_COLS)), importances[order])
plt.xticks(range(len(FEATURE_COLS)), [FEATURE_COLS[i] for i in order], rotation=30, ha='right')
plt.ylabel('Importance')
plt.title('Feature importance')
plt.tight_layout()
plt.savefig('feature_importance.png')
plt.show()
for i in order:
    print(f'  {FEATURE_COLS[i]}: {importances[i]:.3f}')

## 9. Leave-one-session-out cross-validation
Directly relevant to the HR-correlation sign-flip found between your sessions — shows both directions.

In [ ]:
for held_out in session_ids:
    tr_mask, te_mask = df['session_id'] != held_out, df['session_id'] == held_out
    y_tr, y_te = df.loc[tr_mask, TARGET_COL].astype(int), df.loc[te_mask, TARGET_COL].astype(int)

    if y_tr.nunique() < 2 or y_te.nunique() < 2:
        print(f'Held-out {held_out}: skipped (only one class present)')
        continue

    X_tr, X_te = df.loc[tr_mask, FEATURE_COLS], df.loc[te_mask, FEATURE_COLS]
    m = RandomForestClassifier(n_estimators=200, max_depth=5, class_weight='balanced', random_state=42)
    m.fit(X_tr, y_tr)
    auc = roc_auc_score(y_te, m.predict_proba(X_te)[:,1])
    print(f'Held-out {held_out}: AUC = {auc:.3f}  (n={te_mask.sum()}, positive rate {y_te.mean():.2%})')

## 10. Save the model

In [ ]:
joblib.dump(model, 'fatigue_model.pkl')
print('Saved fatigue_model.pkl')
if IN_COLAB:
    files.download('fatigue_model.pkl')
    files.download('feature_importance.png')